In [1]:
import random
import pyspark
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql import SparkSession, functions as F
from datetime import datetime
import ConnectionConfig as cc
from importlib import reload

from pyspark.sql.functions import *

In [2]:
reload(cc)

print(cc.__file__)

cc.setupEnvironment()
cc.listEnvironment()

/home/jovyan/work/02-treasure-hunt-dwh/ConnectionConfig.py
Running in Docker environment.
Setting JAVA_HOME to /usr/lib/jvm/java-17-openjdk-arm64
SHELL: /bin/bash
CONDA_EXE: /opt/conda/bin/conda
_CE_M: 
HOSTNAME: sparkjupyter-data4
LANGUAGE: C.UTF-8
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-arm64
_START_SH_EXECUTED: 1
SPARK_OPTS: --driver-java-options=-Xms1024M --driver-java-options=-Xmx4096M --driver-java-options=-Dlog4j.logLevel=info
NB_UID: 1000
XML_CATALOG_FILES: file:///opt/conda/etc/xml/catalog file:///etc/xml/catalog
PWD: /home/jovyan
CONDA_PREFIX: /opt/conda
PYSPARK_SUBMIT_ARGS: pyspark-shell
HOME: /home/jovyan
LANG: C.UTF-8
NB_GID: 100
CONDA_PROMPT_MODIFIER: (base) 
_CONDA_EXE: /opt/conda/bin/conda
_CONDA_ROOT: /opt/conda
PYTHONPATH: /usr/local/spark/python/lib/py4j-0.10.9.9-src.zip:/usr/local/spark/python:
_CE_CONDA: 
CONDA_SHLVL: 1
SHLVL: 0
CONDA_DIR: /opt/conda
SPARK_HOME: /usr/local/spark
CONDA_PYTHON_EXE: /opt/conda/bin/python
JUPYTER_PORT: 8888
SPARK_CONF_DIR: /usr/local/s

In [3]:
spark = cc.startLocalCluster("all_data")
spark.getActiveSession()

Running in Docker: using pre-installed Spark + JARs


In [5]:
dim_user = spark.read.format("delta").load("spark-warehouse/dim_user")
dim_treasure = spark.read.format("delta").load("spark-warehouse/dim_treasure")
dim_date = spark.read.format("delta").load("spark-warehouse/dim_date")
dim_season = spark.read.format("delta").load("spark-warehouse/dim_season")
dim_weather = spark.read.format("delta").load("spark-warehouse/dim_weather")
fact_treasure_hunt = spark.read.format("delta").load("spark-warehouse/fact_treasure_hunt")

In [6]:
print("Dim User:", dim_user.count())  # 486523
print("Dim Treasure:", dim_treasure.count())  # 250
print("Dim Date:", dim_date.count())  # 1097
print("Dim Season:", dim_season.count())  #4
print("Dim Weather:", dim_weather.count())  #3
print("Fact Treasure Hunt:", fact_treasure_hunt.count())  #1789438

Dim User: 486523
Dim Treasure: 250
Dim Date: 1097
Dim Season: 4
Dim Weather: 3
Fact Treasure Hunt: 1789438


In [8]:
dim_user.show(3, truncate=False)

+-------+-------------------------------------------------+-------------------+------------+----------------+---------+--------------------------+-------+-------+
|user_sk|user_id                                          |city_name          |country_name|experience_level|dedicator|scd_start                 |scd_end|current|
+-------+-------------------------------------------------+-------------------+------------+----------------+---------+--------------------------+-------+-------+
|0      |[00 02 66 4B E2 A6 4D 31 BE C6 3C 26 2B DA 05 B0]|Porbandar Mill Pura|India       |Amateur         |false    |2026-04-21 11:43:38.911634|NULL   |true   |
|4      |[00 05 0C 95 84 B6 45 82 80 D1 7F DC 34 C4 75 60]|Kinhupur           |India       |Amateur         |false    |2026-04-21 11:43:38.911634|NULL   |true   |
|8      |[00 07 66 C4 1B 47 49 59 98 B9 58 A9 66 C2 7A 8A]|Melaka             |Malaysia    |Pirate          |true     |2026-04-21 11:43:38.911634|NULL   |true   |
+-------+-------------

In [9]:
dim_treasure.show(3, truncate=False)

+-----------+----------+-------+-----------+
|treasure_sk|difficulty|terrain|stage_count|
+-----------+----------+-------+-----------+
|0          |0         |0      |1          |
|3          |0         |0      |2          |
|6          |0         |0      |3          |
+-----------+----------+-------+-----------+
only showing top 3 rows


In [10]:
dim_date.show(3, truncate=False)

+-------+----------+---+-----+----+----------+------------+----------+
|date_sk|full_date |day|month|year|month_name|weekday_name|is_weekend|
+-------+----------+---+-----+----+----------+------------+----------+
|0      |2020-09-11|11 |9    |2020|September |Friday      |false     |
|1      |2020-09-12|12 |9    |2020|September |Saturday    |true      |
|2      |2020-09-13|13 |9    |2020|September |Sunday      |true      |
+-------+----------+---+-----+----+----------+------------+----------+
only showing top 3 rows


In [11]:
dim_season.show(3, truncate=False)

+---------+-----------+
|season_SK|season_name|
+---------+-----------+
|1        |Winter     |
|2        |Spring     |
|3        |Summer     |
+---------+-----------+
only showing top 3 rows


In [12]:
dim_weather.show(3, truncate=False)

+----------+---------------+
|weather_sk|name           |
+----------+---------------+
|1         |Rain           |
|2         |No Rain        |
|3         |Weather Unknown|
+----------+---------------+



In [13]:
fact_treasure_hunt.show(3, truncate=False)

+----------------+-------+-----------+-------+---------+----+----------+----------------+
|treasure_hunt_sk|user_sk|treasure_sk|date_SK|season_sk|hour|weather_sk|duration_minutes|
+----------------+-------+-----------+-------+---------+----+----------+----------------+
|3               |465660 |284        |219    |2        |1   |3         |47              |
|7               |465660 |220        |218    |2        |19  |3         |10              |
|11              |465660 |181        |218    |2        |19  |3         |46              |
+----------------+-------+-----------+-------+---------+----+----------+----------------+
only showing top 3 rows


Q1) 1. What is the influence of the user type on the average duration of a treasure hunt? e.g. Does a beginner take longer than an advanced user?


In [14]:


# 1. Join the fact table with the user dimension
# 2. Group by the user's experience level
# 3. Calculate the average duration and count the total hunts for context
# 4. Order the results so the longest average duration is at the top

results_df = fact_treasure_hunt.join(
    dim_user,
    on="user_sk",
    how="inner"
).groupBy(
    "experience_level"
).agg(
    F.avg("duration_minutes").alias("avg_duration_minutes"),
    F.count("treasure_hunt_sk").alias("total_hunts")
).orderBy(
    F.desc("avg_duration_minutes")
)

# Display the final numbers
results_df.show()

+----------------+--------------------+-----------+
|experience_level|avg_duration_minutes|total_hunts|
+----------------+--------------------+-----------+
|    Professional|   70.11131303952749|      90241|
|          Pirate|   70.03229482354091|     654656|
|         Amateur|   70.02740821087923|    1044541|
+----------------+--------------------+-----------+



Q2) Do beginner users on average search for treasures with fewer stages?

In [17]:
# 1. Join the fact table with the user and treasure dimensions
joined_df = fact_treasure_hunt.join(
    dim_user, on="user_sk", how="inner"
).join(
    dim_treasure, on="treasure_sk", how="inner"
)

# ==========================================
# Metric 1: Average stage count by user type
# ==========================================

avg_stages_df = joined_df.groupBy("experience_level").agg(
    F.avg("stage_count").alias("avg_stage_count"),
    F.count("treasure_hunt_sk").alias("total_hunts")
).orderBy(F.asc("avg_stage_count"))

## Average Stage Count by User Type:
avg_stages_df.show()

+----------------+------------------+-----------+
|experience_level|   avg_stage_count|total_hunts|
+----------------+------------------+-----------+
|         Amateur| 3.928435552075026|    1044541|
|          Pirate|3.9294652458695865|     654656|
|    Professional|3.9399386088363384|      90241|
+----------------+------------------+-----------+



In [18]:
# ==========================================
# Metric 2: Frequency of 1, 2, 3, and 4+ stages
# ==========================================
# Frequency of Stage Counts by User Type:

# Categorize the stage counts
categorized_df = joined_df.withColumn(
    "stage_category",
    F.when(F.col("stage_count") == 1, "1 Stage")
     .when(F.col("stage_count") == 2, "2 Stages")
     .when(F.col("stage_count") == 3, "3 Stages")
     .otherwise("4+ Stages")
)

# Group by experience level, pivot by the stage category, and count
frequency_df = categorized_df.groupBy("experience_level").pivot(
    "stage_category", ["1 Stage", "2 Stages", "3 Stages", "4+ Stages"]
).count().fillna(0) # fillna(0) replaces nulls with 0 for categories with no hunts

frequency_df.show()

+----------------+-------+--------+--------+---------+
|experience_level|1 Stage|2 Stages|3 Stages|4+ Stages|
+----------------+-------+--------+--------+---------+
|          Pirate| 244562|   26026|   25865|   358203|
|         Amateur| 391070|   40659|   41331|   571481|
|    Professional|  33615|    3532|    3561|    49533|
+----------------+-------+--------+--------+---------+



Q5) What is the retention rate of users (i.e., how many return after their first treasure found)?

In [19]:

from pyspark.sql.window import Window

# 1. Count the total number of hunts for each user
user_hunt_counts = fact_treasure_hunt.groupBy("user_sk").agg(
    F.count("treasure_hunt_sk").alias("total_hunts")
)

# 2. Assign categories based on your retention definitions
categorized_users = user_hunt_counts.withColumn(
    "engagement_level",
    F.when(F.col("total_hunts") == 1, "1. One-Time User (x = 1)")
     .when((F.col("total_hunts") > 1) & (F.col("total_hunts") <= 3), "2. Returning User (1 < x <= 3)")
     .otherwise("3. Engaged User (x >= 4)")
).withColumn(
    "return_status",
    F.when(F.col("total_hunts") == 1, "Did Not Return (1 hunt)")
     .otherwise("Returned (> 1 hunts)")
)

# Create a window definition to calculate the overall total of users for percentage math
total_users_window = Window.partitionBy()

In [22]:
# ==========================================
# Metric 1: Returned vs. Did Not Return
# ==========================================
returned_vs_not_df = categorized_users.groupBy("return_status").agg(
    F.count("user_sk").alias("user_count")
).withColumn(
    "percentage_share",
    F.round((F.col("user_count") / F.sum("user_count").over(total_users_window)) * 100, 2)
)

# High-Level Retention Rate (Returned vs. Did Not Return):
returned_vs_not_df.show(truncate=False)

+-----------------------+----------+----------------+
|return_status          |user_count|percentage_share|
+-----------------------+----------+----------------+
|Did Not Return (1 hunt)|21721     |4.47            |
|Returned (> 1 hunts)   |464619    |95.53           |
+-----------------------+----------+----------------+



In [23]:
# ==========================================
# Metric 2: Detailed Engagement Transition
# ==========================================
engagement_breakdown_df = categorized_users.groupBy("engagement_level").agg(
    F.count("user_sk").alias("user_count")
).withColumn(
    "percentage_share",
    F.round((F.col("user_count") / F.sum("user_count").over(total_users_window)) * 100, 2)
).orderBy("engagement_level")

# Detailed Engagement Breakdown (One-Time -> Returning -> Engaged):
engagement_breakdown_df.show(truncate=False)

+------------------------------+----------+----------------+
|engagement_level              |user_count|percentage_share|
+------------------------------+----------+----------------+
|1. One-Time User (x = 1)      |21721     |4.47            |
|2. Returning User (1 < x <= 3)|409231    |84.15           |
|3. Engaged User (x >= 4)      |55388     |11.39           |
+------------------------------+----------+----------------+

